In [0]:
import mlflow
from mlflow import MlflowClient

client = MlflowClient()

# Buscar el experimento control-2
usuario = spark.sql("SELECT current_user()").collect()[0][0]
experiment = mlflow.get_experiment_by_name(f"/Users/{usuario}/control-2")

# Traer todas las corridas ordenadas por AUC descendente
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.auc DESC"]
)

best_run = runs[0]
print(f"Mejor corrida: {best_run.data.tags['mlflow.runName']}")
print(f"AUC: {best_run.data.metrics['auc']:.4f}")

# Registrar el modelo de esa corrida en el Model Registry
model_uri = f"runs:/{best_run.info.run_id}/model"
registered_model_name = "wine_quality_model"   # o "catalogo.esquema.wine_quality_model" si usas Unity Catalog

result = mlflow.register_model(model_uri=model_uri, name=registered_model_name)

print(f"Modelo registrado: {result.name}, versión {result.version}")

# Asignar el alias PRINCIPAL a esa versión
client.set_registered_model_alias(
    name=registered_model_name,
    alias="PRINCIPAL",
    version=result.version
)

print(f"Alias 'PRINCIPAL' asignado a la versión {result.version} de {registered_model_name}")